# SafeRide Vision — Google Colab

This notebook is the supported way to run the project. It clones the code from GitHub, installs its requirements, loads every model weight from Google Drive or a Colab upload, then starts the API.

Before running, set `REPO_URL` below. Put weights in the Drive folder configured by `config.py` or upload the missing files when prompted.

In [ ]:
# GitHub repository containing this project
REPO_URL = "https://github.com/Salal04/Safe-Ride.git"  # replace this
BRANCH = "main"
REPO_DIR = "/content/deepSort"

import subprocess
from pathlib import Path
if "YOUR_ACCOUNT" in REPO_URL or "YOUR_REPOSITORY" in REPO_URL:
    raise ValueError("Replace REPO_URL with your GitHub repository URL.")
repo = Path(REPO_DIR)
if (repo / ".git").exists():
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    if repo.exists() and any(repo.iterdir()):
        raise RuntimeError(f"{REPO_DIR} exists and is not a Git checkout.")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
print("Repository ready:", REPO_DIR)

In [ ]:
# Mount Google Drive, then install the project dependencies.
from google.colab import drive
drive.mount("/content/drive")
%cd /content/deepSort
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt

## Load model weights

All model paths and filenames come from `config.py`. By default, checkpoints are stored in `MyDrive/SafeRideVision/weights`. Copy them there before running, or select all missing files in the upload dialog. Upload filenames must match the configured basenames.

In [ ]:
import sys
from pathlib import Path
from google.colab import files

sys.path.insert(0, REPO_DIR)
import config

required = config.REQUIRED_WEIGHTS
missing = [(name, Path(path)) for name, path in required.items() if not Path(path).is_file()]
if missing:
    print("Upload these missing weight files:")
    for name, path in missing:
        print(f"  {name}: {path.name}")
    uploaded = files.upload()
    for name, destination in missing:
        filename = destination.name
        if filename not in uploaded:
            raise FileNotFoundError(f"Upload the configured filename exactly: {filename}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(uploaded[filename])
        print(f"Saved {name} to {destination}")

for name, path in required.items():
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty model weight '{name}': {path}")
    print(f"Ready: {name} -> {path} ({path.stat().st_size:,} bytes)")

## Start the backend

If ngrok is needed, enter your own token privately when prompted. It is not stored in the notebook or source code.

In [ ]:
# Optional ngrok token. Leave blank to use the anonymous tunnel.
from getpass import getpass
from pyngrok import conf

token = getpass("Ngrok auth token (leave blank to skip): ")
if token.strip():
    conf.get_default().auth_token = token.strip()

%cd /content/deepSort
!python backend_server.py